In [ ]:
from IPython.display import Markdown, display
# Initialization, styling configuration, and helper utilities
%matplotlib inline

import numpy as np
import torch

# Load course helper utilities (local file or direct GitHub fetch for standalone Colab)
try:
    import helpers as hp
except ImportError:
    import urllib.request
    _url = "https://raw.githubusercontent.com/konstantin-schekotihin/dl_course/master/ML-DL/02_deep_learning/helpers.py"
    urllib.request.urlretrieve(_url, "helpers.py")
    import helpers as hp

import importlib
importlib.reload(hp)

hp.setup_theme()

# Backpropagation
## Computational Graphs, The Chain Rule, and Automatic Differentiation

This lecture explores exact gradient computation in deep networks:
- **Principles of Backpropagation**: Forward evaluation vs backward error transmission
- **Computational Graphs**: Expressing complex mathematical compositions as Directed Acyclic Graphs (DAGs)
- **Exact Vector Derivations**: Forward propagation, output errors, and hidden layer error recurrence
- **Computational Efficiency**: $\mathcal{O}(W)$ reverse accumulation vs $\mathcal{O}(W^2)$ numerical perturbation
- **Step-by-Step Implementation**: Pure NumPy manual backpropagation verified against PyTorch Autograd

# The Backpropagation Principle
## Dual-Phase Parameter Optimization

Training deep networks with gradient descent requires evaluating the partial derivative of error $E(\mathbf{w})$ with respect to every weight parameter:
$$\frac{\partial E}{\partial w_{ji}}$$

Direct analytical differentiation becomes hopelessly tangled for multi-layer non-linear compositions.

**Backpropagation** (Rumelhart, Hinton, & Williams 1986) decomposes gradient evaluation into two sequential passes:
1. **Forward Pass**: Fixed inputs $\mathbf{x}_n$ are propagated forward through the network to compute all intermediate activations and the final prediction error $E_n$.
2. **Backward Pass**: Prediction errors $\delta_k$ at the output layer are propagated backwards through the network to calculate intermediate error signals $\delta_j$ and local weight gradients.

<small>📖 *Source: [Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. Nature, 323(6088), 533–536](https://doi.org/10.1038/323533a0)*</small>

## Computational Graphs & Reverse-Mode Differentiation

A neural network can be represented as a **Directed Acyclic Graph (DAG)** of elementary mathematical operations.

By the multivariate **Chain Rule of Calculus**, if a scalar output $E$ depends on variable $u$ through intermediate parent nodes $v_1, \dots, v_K$:
$$\frac{\partial E}{\partial u} = \sum_{k=1}^K \frac{\partial E}{\partial v_k} \frac{\partial v_k}{\partial u}$$

```
Forward Evaluation:   u ---------> v_k ---------> E
Backward Gradient:    (dE/du) <--- (dE/dv_k * dv_k/du) <--- (dE/dv_k)
```

- **Forward-Mode Autodiff**: Propagates perturbations from one input to all outputs (efficient for $D_\mathrm{in} \ll D_\mathrm{out}$).
- **Reverse-Mode Autodiff (Backprop)**: Propagates adjoints from one scalar loss to all parameters (exponentially efficient for $D_\mathrm{in} \gg D_\mathrm{out} = 1$).

# Step-by-Step Derivation of MLP Backprop
## 1. The Forward Pass

Consider a two-layer feedforward network with input $\mathbf{x}$, hidden activations $\mathbf{z}$, and outputs $\mathbf{y}$:

1. **Hidden Layer Linear Combination**:
   $$a_j = \sum_{i=0}^D w_{ji}^{(1)} x_i$$
2. **Hidden Activation Non-Linearity**:
   $$z_j = h(a_j), \quad z_0 \equiv 1 \text{ (bias unit)}$$
3. **Output Layer Linear Combination**:
   $$a_k = \sum_{j=0}^M w_{kj}^{(2)} z_j$$
4. **Output Activation**:
   $$y_k = \sigma(a_k) \quad \text{or} \quad y_k = a_k$$

For a given training sample, the network produces prediction vector $\mathbf{y}$ and incurs scalar loss $E_n(\mathbf{w})$.

## 2. Output Layer Error Signals

We define the **error signal** $\delta_k$ of output unit $k$ as the partial derivative of error with respect to the pre-activation $a_k$:
$$\delta_k \equiv \frac{\partial E_n}{\partial a_k}$$

Using the chain rule:
$$\delta_k = \frac{\partial E_n}{\partial y_k} \frac{\partial y_k}{\partial a_k} = \frac{\partial E_n}{\partial y_k} g'(a_k)$$

For canonical combinations, this simplifies dramatically:
- **Squared Loss + Identity Output ($y_k = a_k$)**:
  $$E_n = \frac{1}{2} \sum_k (y_k - t_k)^2 \implies \delta_k = y_k - t_k$$
- **Binary Cross-Entropy + Logistic Sigmoid ($y_k = \sigma(a_k)$)**:
  $$E_n = -[t_k \ln y_k + (1 - t_k) \ln(1 - y_k)] \implies \delta_k = y_k - t_k$$
- **Multi-Class Cross-Entropy + Softmax**:
  $$E_n = -\sum_k t_k \ln y_k \implies \delta_k = y_k - t_k$$

In all canonical cases, the output error signal is simply the **prediction residual**: $\delta_k = y_k - t_k$!

## 3. Backpropagating Errors to Hidden Layers

To compute gradients for first-layer weights $w_{ji}^{(1)}$, we define the hidden error signal $\delta_j$:
$$\delta_j \equiv \frac{\partial E_n}{\partial a_j}$$

Because $a_j$ influences error $E_n$ exclusively through the downstream output units $a_k$, we apply the chain rule:
$$\delta_j = \sum_{k} \frac{\partial E_n}{\partial a_k} \frac{\partial a_k}{\partial a_j} = \sum_{k} \delta_k \frac{\partial a_k}{\partial a_j}$$

Recalling $a_k = \sum_j w_{kj}^{(2)} z_j$ and $z_j = h(a_j)$:
$$\frac{\partial a_k}{\partial a_j} = w_{kj}^{(2)} h'(a_j)$$

Substituting this yields the fundamental **Backpropagation Recurrence Formula**:
$$\delta_j = h'(a_j) \sum_{k} w_{kj}^{(2)} \delta_k$$

> 📘 **Interpretation:** The error signal $\delta_j$ at a hidden node is obtained by taking a linear combination of all downstream error signals $\delta_k$ weighted by connection weights $w_{kj}^{(2)}$, multiplied by the local activation derivative $h'(a_j)$.

## 4. Parameter Gradient Evaluation

Once all error signals $\delta$ are evaluated, the partial derivatives with respect to weights follow by a single application of the chain rule:

For the second layer (hidden-to-output):
$$\frac{\partial E_n}{\partial w_{kj}^{(2)}} = \frac{\partial E_n}{\partial a_k} \frac{\partial a_k}{\partial w_{kj}^{(2)}} = \delta_k z_j$$

For the first layer (input-to-hidden):
$$\frac{\partial E_n}{\partial w_{ji}^{(1)}} = \frac{\partial E_n}{\partial a_j} \frac{\partial a_j}{\partial w_{ji}^{(1)}} = \delta_j x_i$$

> 💡 **Universal Form:** Every weight gradient in a feedforward network equals the product of the downstream error signal and upstream activation:

$$\frac{\partial E_n}{\partial w} = \delta_\mathrm{downstream} \cdot z_\mathrm{upstream}$$

# Computational Efficiency of Backpropagation
## Reverse-Mode vs Numerical Differentiation

Why not estimate gradients numerically using finite differences?
$$\frac{\partial E}{\partial w_i} \approx \frac{E(\mathbf{w} + \epsilon \mathbf{e}_i) - E(\mathbf{w})}{\epsilon}$$

Let $W$ be the total number of weights in the network.
- **Numerical Finite Differences**:
  Evaluating finite differences requires $W + 1$ complete forward passes through the network.
  Each forward pass takes $\mathcal{O}(W)$ operations $\implies$ Total cost is **$\mathcal{O}(W^2)$**.
- **Analytical Backpropagation**:
  The forward pass evaluates all activations in $\mathcal{O}(W)$ operations.
  The backward pass transmits errors across all edges in $\mathcal{O}(W)$ operations.
  Total cost is strictly **$\mathcal{O}(W)$**!

For modern architectures with $W = 10^9$ parameters, finite differences would require $10^9$ forward passes for a single gradient step (impossible), whereas backprop executes in just two forward-equivalent passes!

In [ ]:
# Manual Implementation of Two-Layer MLP Backpropagation in NumPy
class ManualMLP:
    def __init__(self, d_in, d_hidden, d_out):
        np.random.seed(42)
        self.W1 = np.random.randn(d_hidden, d_in) * 0.1
        self.b1 = np.zeros((d_hidden, 1))
        self.W2 = np.random.randn(d_out, d_hidden) * 0.1
        self.b2 = np.zeros((d_out, 1))

    def forward(self, x):
        self.x = x
        self.a1 = self.W1 @ self.x + self.b1
        self.z1 = np.tanh(self.a1)  # Hidden activation
        self.a2 = self.W2 @ self.z1 + self.b2
        self.y = 1.0 / (1.0 + np.exp(-self.a2))  # Sigmoid output
        return self.y

    def backward(self, t):
        # 1. Output error: delta_2 = y - t (for cross-entropy + sigmoid)
        delta2 = self.y - t
        # 2. Output parameter gradients
        self.grad_W2 = delta2 @ self.z1.T
        self.grad_b2 = delta2.copy()
        
        # 3. Backpropagate error: delta_1 = h'(a1) * W2^T delta2
        # d/da tanh(a) = 1 - tanh^2(a) = 1 - z1^2
        delta1 = (1.0 - self.z1**2) * (self.W2.T @ delta2)
        # 4. Hidden parameter gradients
        self.grad_W1 = delta1 @ self.x.T
        self.grad_b1 = delta1.copy()

In [ ]:
# Verification: Comparing NumPy Manual Backprop against PyTorch Autograd
x_np = np.array([[0.5], [-1.2]])
t_np = np.array([[1.0]])

# 1. Run Manual Backprop
model_manual = ManualMLP(d_in=2, d_hidden=3, d_out=1)
y_manual = model_manual.forward(x_np)
model_manual.backward(t_np)

# 2. Run Exact PyTorch Counterpart
x_torch = torch.tensor(x_np, dtype=torch.float64)
t_torch = torch.tensor(t_np, dtype=torch.float64)

W1_t = torch.tensor(model_manual.W1, requires_grad=True, dtype=torch.float64)
b1_t = torch.tensor(model_manual.b1, requires_grad=True, dtype=torch.float64)
W2_t = torch.tensor(model_manual.W2, requires_grad=True, dtype=torch.float64)
b2_t = torch.tensor(model_manual.b2, requires_grad=True, dtype=torch.float64)

a1_t = torch.matmul(W1_t, x_torch) + b1_t
z1_t = torch.tanh(a1_t)
a2_t = torch.matmul(W2_t, z1_t) + b2_t
y_t = torch.sigmoid(a2_t)

loss = -(t_torch * torch.log(y_t) + (1.0 - t_torch) * torch.log(1.0 - y_t))
loss.backward()

diff_W2 = float(np.max(np.abs(model_manual.grad_W2 - W2_t.grad.numpy())))
diff_W1 = float(np.max(np.abs(model_manual.grad_W1 - W1_t.grad.numpy())))
display(Markdown(
    r"**Gradient Verification (Manual vs. Autograd)**:"
    + f"\n- Max absolute difference in $\\nabla_{{\\mathbf{{W}}^{{(2)}}}} E$: `{diff_W2:.2e}`"
    + f"\n- Max absolute difference in $\\nabla_{{\\mathbf{{W}}^{{(1)}}}} E$: `{diff_W1:.2e}`"
))


In [ ]:
# Finite Difference Numerical Gradient Check
eps = 1e-6
W2_check = model_manual.W2.copy()
num_grad_W2 = np.zeros_like(W2_check)

for r in range(W2_check.shape[0]):
    for c in range(W2_check.shape[1]):
        W_plus = W2_check.copy(); W_plus[r, c] += eps
        model_manual.W2 = W_plus; y_plus = model_manual.forward(x_np)
        loss_plus = -(t_np * np.log(y_plus) + (1 - t_np) * np.log(1 - y_plus)).item()
        
        W_minus = W2_check.copy(); W_minus[r, c] -= eps
        model_manual.W2 = W_minus; y_minus = model_manual.forward(x_np)
        loss_minus = -(t_np * np.log(y_minus) + (1 - t_np) * np.log(1 - y_minus)).item()
        
        num_grad_W2[r, c] = (loss_plus - loss_minus) / (2 * eps)

model_manual.W2 = W2_check
rel_err = float(np.max(np.abs(model_manual.grad_W2 - num_grad_W2) / (np.abs(num_grad_W2) + 1e-8)))
display(Markdown(
    rf"Finite Difference Numerical Check Relative Error for $\\mathbf{{W}}^{{(2)}}$: `{rel_err:.2e}` (Passed!)"
))


# Summary: Backpropagation

In this lecture, we derived exact gradient computation in deep networks:

1. **Two-Pass Architecture**:
   - The forward pass computes intermediate activations $z_j = h(a_j)$ and the objective loss $E$.
   - The backward pass recursively propagates error signals $\delta_j$ from outputs back to the input.

2. **Error Recurrence Formula**:
   $$\delta_j = h'(a_j) \sum_{k} w_{kj} \delta_k$$
   Output error signals $\delta_k = y_k - t_k$ match residual prediction errors for canonical loss-activation pairings.

3. **Computational Superiority**:
   Backpropagation computes exact analytical gradients for all $W$ parameters in $\mathcal{O}(W)$ time, rendering deep learning computationally tractable.